# Lab 06: Sequence-Level and Black-Box Distillation

**Tier 2 lab.** Part A executes and asserts anywhere; Part B trains only when
`RUN_TRAINING = True`.

**The question.** Labs 03-05 assumed you could open the teacher and read its logits, the raw
pre-softmax scores it assigns to every vocabulary token. That setting is called **white-box**
distillation: you can see inside the teacher. Often you cannot. The teacher is behind an API,
or its tokenizer differs from the student's so its per-token scores do not line up with your
token ids, or the serving stack returns text and nothing else. That setting is **black-box**
distillation: the only thing the teacher gives you is its output text. What remains is
**distilling from the teacher's outputs**, in two standard forms:

- **SeqKD** (sequence-level knowledge distillation, Kim & Rush 2016): the teacher *generates*
  full completions for your prompts, and the student is fine-tuned on those completions with
  plain cross-entropy, the same loss as ordinary supervised training. The insight that made
  this famous: training on the teacher's *mode* (its single most likely output for each
  prompt) approximates matching its sequence-level distribution, and for translation it
  worked nearly as well as token-level KD at a fraction of the complexity.
- **Trace SFT** (the DeepSeek-R1-Distill pattern): identical mechanics, but the "teacher
  output" is a published corpus of reasoning traces. A trace is the step-by-step text a
  reasoning model writes out before its final answer, its worked solution. Here there are no
  logits, no reinforcement learning, and no teacher access at all: you run supervised
  fine-tuning (SFT) on someone else's teacher-decode, purchased as data.

Mechanically both are just Lab 03's `hard` arm run on different text. **The entire content of
this lab is economic and behavioral.** The economics come first because on bandwidth-limited
hardware (hardware where the bottleneck is how fast weights can be read from memory, not how
fast arithmetic runs) they are brutal: SeqKD is the one workload where the *large* model
decodes, meaning it generates new tokens one at a time, and every generated token requires
reading all of the model's weights from memory once. The README's workload table calls it the
expensive pattern; Part A prices it to the hour before Part B is allowed to generate a single
token. The behavioral question, namely what a student learns from samples that it would have
learned differently from logits, is what Part C's comparison against Lab 04's cached-logit
student answers.

In [1]:
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")   # widget progress bars crash some notebook stacks; plain logs are fine

import sys, os, json, math, time
sys.path.insert(0, "../code")

import torch
import torch.nn.functional as F

from kd_core import (shift_for_next_token, completion_mask_from_prompt_lens,
                     top1_agreement, mean_entropy, distinct_n, self_bleu,
                     kl_divergence, masked_mean)
from kd_pipeline import (set_seed_everywhere, config_fingerprint, MemoryPlan,
                         infer_gb, full_ft_gb, bandwidth_bound_decode_tps,
                         decode_wallclock_hours, RunManifest)

RUN_TRAINING = False        # <-- flip on the training box
SEED = 17
set_seed_everywhere(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {device} | RUN_TRAINING: {RUN_TRAINING}")

CFG = dict(
    gen_teacher="HuggingFaceTB/SmolLM2-1.7B-Instruct",   # deliberately small: it must decode
    student="HuggingFaceTB/SmolLM2-360M-Instruct",
    trace_dataset="HuggingFaceTB/smol-smoltalk",  # itself a teacher-generated corpus
    data="../data/lab03", seq_len=384, gen_max_new=256, n_gen_prompts=2048,
    lr=3e-5, batch_size=8, grad_accum=4, max_steps=1500, warmup_steps=50,
)

torch 2.13.0+cpu | device: cpu | RUN_TRAINING: False


## Part A · 1: Price the decode before anything generates

Two words of serving vocabulary first. **Prefill** is scoring a prompt the model already has:
all of the prompt's tokens are processed in one parallel pass, so the weights are read from
memory once for the whole prompt. **Decode** is generating new tokens one at a time: each new
token depends on the token before it, so the model must run one full forward pass per token,
and each pass streams every weight from memory again. That is why decode speed is limited by
memory bandwidth, the number of bytes per second the hardware can read from memory, rather
than by arithmetic. The resulting ceiling is called a **roofline**, or a bandwidth bound:
decode throughput, measured in tokens per second (tok/s), cannot exceed
`bandwidth / bytes-per-token`, because every generated token streams every weight once.
`kd_pipeline`'s `bandwidth_bound_decode_tps` computes exactly that ceiling.

The table below prices generating this course's corpus (2,048 prompts × 256 new tokens each,
which multiplies out to roughly 0.5M tokens) for four teacher sizes on a 273 GB/s bandwidth
budget, and then asserts the design decision the prices force.

Read the last column the way a buyer would, remembering that the roofline is *per stream*,
one sequence generated at a time. Batching, generating many prompts at once, amortizes the
weight reads across the whole batch, because a single read of the weights serves every
sequence in the batch. So real wall-clock divides by an order of magnitude when you generate
16-32 prompts at once. The 1.7B teacher prices at a couple of single-stream hours, which
batching turns into tens of minutes: affordable, and the reason `CFG["gen_teacher"]` is 1.7B.
The 32B teacher prices at *days* single-stream. Batching brings it to hours, but it remains a
different economic object. That is why serious SeqKD from big teachers is a **purchased
asset**: you generate once and archive the result, or you use someone's published generation
(that is what the trace dataset is), or you rent an hour of fat-GPU time for exactly this
phase.

The cell also checks two memory plans, one per Part B phase. The generation plan budgets for
the teacher's weights plus its KV cache: during generation the model stores the attention keys
and values for every token it has seen so far, so that it does not recompute them for each new
token, and that store grows with sequence length and with how many sequences run at once.

In [2]:
corpus_tokens = CFG["n_gen_prompts"] * CFG["gen_max_new"]
BW = 273.0
print(f"corpus to generate: {corpus_tokens/1e6:.2f} M tokens\n")
print(f"{'teacher':>10} {'bf16 GB':>8} {'roofline tok/s':>15} {'hours (1 stream)':>17}")
rows = {}
for pb in (0.36, 1.7, 8.0, 32.0):
    tps = bandwidth_bound_decode_tps(pb, BW)
    hrs = decode_wallclock_hours(corpus_tokens, tps)
    rows[pb] = hrs
    print(f"{pb:>9}B {infer_gb(pb):>8.1f} {tps:>15.1f} {hrs:>17.2f}")

assert rows[1.7] < 2.5, "1.7B: couple of single-stream hours -> batched tens of minutes"
assert rows[32.0] > 10 * rows[1.7], "the 32B generator is a different economic object"
assert rows[32.0] > 24, "32B single-stream: days-scale — never pay this twice"

# And the memory plans for both Part B phases:
gen_plan = MemoryPlan(total_gb=128.0).add("teacher 1.7B bf16 + KV", infer_gb(1.7) + 6.0)
sft_plan = MemoryPlan(total_gb=128.0).add("student 360M full FT + activ.", full_ft_gb(0.36) + 4.0)
gen_plan.assert_fits(); sft_plan.assert_fits()
print("\ndecode priced; generation and SFT phases both fit with room to spare")

corpus to generate: 0.52 M tokens

   teacher  bf16 GB  roofline tok/s  hours (1 stream)
     0.36B      0.7           379.2              0.38
      1.7B      3.4            80.3              1.81
      8.0B     16.0            17.1              8.54
     32.0B     64.0             4.3             34.14

decode priced; generation and SFT phases both fit with room to spare


## Part A · 2: The library path, introspected rather than trusted

TRL's GKD trainer has a `seq_kd` flag that does exactly this lab's Part B: the teacher
generates, and the student is fine-tuned on the generations. The same config object carries
two knobs this course keeps returning to: `lmbda`, the fraction of training data that is
generated by the student itself rather than taken from a fixed corpus, and `beta`, which
selects the divergence being minimized. Lab 07 sweeps both; here they matter only as fields
whose existence we verify. This course's own README once described a TRL parameter that no
longer exists in the installed version. Documentation drifts, and that drift was caught by
*checking the installed object*, not by re-reading the docs. So that check is now a
pre-flight cell: assert at runtime that the fields this lab's plans depend on exist, and
print their defaults for the record. If this cell fails after a TRL upgrade, the lab tells
you before a run does.

In [3]:
import dataclasses
from trl.experimental.gkd import GKDConfig

fields = {f.name: f.default for f in dataclasses.fields(GKDConfig)}
required = ["seq_kd", "lmbda", "beta", "temperature", "max_new_tokens",
            "teacher_model_name_or_path"]
missing = [k for k in required if k not in fields]
assert not missing, f"TRL drift: GKDConfig lost {missing} — re-ground this lab before running"
for k in required:
    print(f"GKDConfig.{k:<28} default = {fields[k]}")
print("\nfields present. seq_kd=True + lmbda=... is the library's SeqKD;")
print("Part B also hand-rolls the generation loop once, because you should see it.")

/tmp/ipykernel_11247/12684827.py:2: TRLExperimentalWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  from trl.experimental.gkd import GKDConfig
/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GKDConfig.seq_kd                       default = False
GKDConfig.lmbda                        default = 0.5
GKDConfig.beta                         default = 0.5
GKDConfig.temperature                  default = 0.9
GKDConfig.max_new_tokens               default = 128
GKDConfig.teacher_model_name_or_path   default = None

fields present. seq_kd=True + lmbda=... is the library's SeqKD;
Part B also hand-rolls the generation loop once, because you should see it.


## Part A · 3: Audit the purchased asset

Trace SFT's whole risk is that you train on data you did not produce. So audit it the way you
audited your own tensors in Lab 03: role structure (are the conversations well-formed
sequences of system, user, and assistant turns), length distribution, and, the check specific
to bought corpora, a *fingerprint of provenance*: does the data look like single
teacher-generated turns, or like scraped mixtures from many sources? This cell streams a
slice of the trace corpus and asserts the structural facts Part B relies on. (SmolTalk is
itself generated by larger teacher models, which makes it a published SeqKD corpus in exactly
this lab's sense.)

In [4]:
from datasets import load_dataset

stream = load_dataset(CFG["trace_dataset"], split="train", streaming=True)
sample, n = [], 512
for ex in stream:
    sample.append(ex["messages"])
    if len(sample) >= n:
        break

roles_ok = sum(1 for m in sample
               if len(m) >= 2 and m[-1]["role"] == "assistant"
               and all(t["role"] in ("system", "user", "assistant") for t in m))
lens = sorted(len(m[-1]["content"]) for m in sample)
p50, p95 = lens[len(lens)//2], lens[int(len(lens)*0.95)]

print(f"rows audited: {n} | well-formed role structure: {roles_ok}/{n}")
print(f"assistant-turn length: p50 {p50} chars, p95 {p95} chars")
assert roles_ok / n > 0.95, "corpus structure is not what Part B assumes"
assert p95 < 20_000, "pathological lengths present; add a length filter before SFT"
print("purchased-asset audit passed")

rows audited: 512 | well-formed role structure: 511/512
assistant-turn length: p50 1174 chars, p95 2526 chars
purchased-asset audit passed


## Part B: Generate, then fine-tune; or skip straight to the traces

Three artifacts come out of Part B, all trained with the *same* SFT loop (Lab 03's `kd_step`
at `alpha=0`, meaning hard cross-entropy on the text, with no logits anywhere):

- `seqkd`: student SFT'd on the 1.7B teacher's own generations for Lab 03's prompts.
- `trace-sft`: student SFT'd on an equal-sized slice of the published corpus.
- (comparison) Lab 04's cached-logit student, already on disk, same prompt set.

One mechanical note on the loop: it uses gradient accumulation, which means the gradients
from several small batches are added up before a single optimizer step, simulating a batch
larger than fits in memory at once.

The generation loop is written out rather than hidden behind `seq_kd=True` so you can see the
two decisions that shape SeqKD quality. First, **sampling temperature**: greedy decoding
(always take the single most likely next token) gives the teacher's mode, which is classical
SeqKD; sampling at temperature T=1 draws tokens from the teacher's full distribution, which
is noisier but more diverse. Second, **EOS handling**, where EOS is the special
end-of-sequence token a model emits to mark that it is finished. A generation that hits the
length cap without ever emitting EOS is a completion with no ending, and a student trained on
such data learns to never stop. Those generations are filtered out, and the filter rate is
logged because it is a quality signal.

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup

def generate_corpus(cfg, greedy=True):
    tok = AutoTokenizer.from_pretrained(cfg["gen_teacher"])
    teacher = AutoModelForCausalLM.from_pretrained(
        cfg["gen_teacher"], dtype=torch.bfloat16).to(device).eval()
    tr = torch.load(os.path.join(cfg["data"], "train.pt"))
    outs, dropped, t0 = [], 0, time.time()
    for i in range(cfg["n_gen_prompts"]):
        plen = tr["prompt_lens"][i]
        prompt = tr["input_ids"][i:i+1, :plen].to(device)
        with torch.no_grad():
            gen = teacher.generate(
                prompt, do_sample=not greedy, temperature=None if greedy else 1.0,
                max_new_tokens=cfg["gen_max_new"], pad_token_id=tok.eos_token_id)
        comp = gen[0, plen:]
        if tok.eos_token_id not in comp:      # hit the cap: teaches never-stop; drop it
            dropped += 1
            continue
        outs.append({"prompt_len": int(plen), "ids": gen[0].cpu()})
        if (i + 1) % 200 == 0:
            rate = sum(len(o["ids"]) - o["prompt_len"] for o in outs) / (time.time() - t0)
            print(f"  {i+1}/{cfg['n_gen_prompts']} prompts, {rate:,.0f} tok/s, "
                  f"dropped {dropped} capped generations")
    del teacher
    if device == "cuda":
        torch.cuda.empty_cache()
    print(f"generated {len(outs)} completions, dropped {dropped} "
          f"({dropped/cfg['n_gen_prompts']:.1%} hit the length cap)")
    torch.save(outs, "../data/lab06_seqkd_corpus.pt")
    return outs

def pack(outs, cfg, tok):
    ids = torch.full((len(outs), cfg["seq_len"]), tok.convert_tokens_to_ids("<|endoftext|>"),
                     dtype=torch.long)
    plens = []
    for i, o in enumerate(outs):
        seq = o["ids"][:cfg["seq_len"]]
        ids[i, :len(seq)] = seq
        plens.append(o["prompt_len"])
    mask = completion_mask_from_prompt_lens(ids, plens,
                                            pad_token_id=int(ids[0, -1]))
    labels = ids.clone(); labels[~mask] = -100
    return ids, labels, mask

def sft(name, ids, labels, mask, cfg):
    set_seed_everywhere(SEED)
    student = AutoModelForCausalLM.from_pretrained(cfg["student"], dtype=torch.bfloat16).to(device)
    opt = torch.optim.AdamW(student.parameters(), lr=cfg["lr"])
    sched = get_cosine_schedule_with_warmup(opt, cfg["warmup_steps"], cfg["max_steps"])
    step = 0
    while step < cfg["max_steps"]:
        for i in range(0, len(ids), cfg["batch_size"]):
            b_ids = ids[i:i+cfg["batch_size"]].to(device)
            b_lab = labels[i:i+cfg["batch_size"]].to(device)
            logits = student(b_ids).logits
            loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.shape[-1]),
                                   b_lab[:, 1:].reshape(-1), ignore_index=-100)
            (loss / cfg["grad_accum"]).backward()
            if (step + 1) % cfg["grad_accum"] == 0:
                torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad()
            if step % 200 == 0:
                print(f"[{name}] step {step:>5}  CE {float(loss):.3f}")
            step += 1
            if step >= cfg["max_steps"]:
                break
    out = f"../runs/lab06/{name}_{config_fingerprint({**cfg, 'seed': SEED, 'arm': name})}"
    os.makedirs(out, exist_ok=True)
    student.save_pretrained(out)
    RunManifest(name=name, config=cfg, seed=SEED,
                artifacts_out={"checkpoint": out}).save(out)
    del student
    if device == "cuda":
        torch.cuda.empty_cache()
    return out

if RUN_TRAINING:
    tok = AutoTokenizer.from_pretrained(CFG["student"])
    outs = generate_corpus(CFG, greedy=True)                       # the expensive phase
    ids, labels, mask = pack(outs, CFG, tok)
    seqkd_ckpt = sft("seqkd", ids, labels, mask, CFG)
    # trace arm: Lab 03's tensors ARE a packed slice of the trace corpus already
    tr = torch.load(os.path.join(CFG["data"], "train.pt"))
    trace_ckpt = sft("trace-sft", tr["input_ids"], tr["labels"], tr["mask"], CFG)
    print("checkpoints:", seqkd_ckpt, trace_ckpt)
else:
    print("RUN_TRAINING=False — Part B compiled but did not execute.")
    print("Generation is the priced phase from A·1 (~an hour); each SFT is Lab 03-scale.")

RUN_TRAINING=False — Part B compiled but did not execute.
Generation is the priced phase from A·1 (~an hour); each SFT is Lab 03-scale.


## Part C: The verdict, quality per unit of teacher compute

Evaluate all three students (`seqkd`, `trace-sft`, and Lab 04's `cached`) with the same
diagnostics (agreement and KL against the 1.7B teacher on held-out data, entropy, distinct-3),
then divide by what each cost in teacher compute. In the table, remember A·1's vocabulary:
prefill is the cheap parallel scoring pass over existing text, decode is the expensive
token-by-token generation pass.

| student | teacher cost | expected standing |
|---|---|---|
| Lab 04 cached-logit | prefill only (minutes) | best agreement/KL, because it saw the full distribution |
| `seqkd` | **decode** (the A·1 price) | close on in-domain behavior; lower entropy (it learned the teacher's *mode*, not its spread) |
| `trace-sft` | zero (purchased) | most domain-shifted; competitive where the traces match your prompts, weakest where they don't |

**Expected ranges.** Cached-logit beats greedy SeqKD on agreement by 1-5 points at equal
steps. The reason: logits carry per-token "dark knowledge", the teacher's full ranking of
every plausible alternative token together with how probable each one is, and a sampled
completion shows only the one token that was actually chosen, so samples cannot carry that
information. SeqKD's entropy runs 0.1-0.4 nats *below* both others; a nat is the unit of
entropy when the logarithm is natural, and lower entropy means the student's output
distribution is more narrowly concentrated. The cause is that greedy generation is a
mode-seeking data source: it only ever shows the student the teacher's single most likely
continuation, which is an echo of Lab 05's β=1 column, arrived at through the data instead of
through the loss. If `trace-sft` wins agreement against the teacher, be suspicious: it likely
means your held-out prompts resemble the trace corpus more than they resemble your actual use
case.

**Failure signatures.**

- *SeqKD student never emits EOS in its own generations.* The cap-filter in `generate_corpus`
  was skipped, or the drop rate was high and ignored. That printed drop-rate is a
  load-bearing number.
- *Trace-SFT student is fluent but wrong in a confident register.* This is style transfer
  without capability transfer, the classic black-box failure: the student learned how the
  teacher sounds without learning what the teacher knows. It shows up in ECE (expected
  calibration error, the gap between how confident the model's probabilities are and how
  often it is actually right) before it shows up in reading the outputs; check calibration
  first.
- *All three within noise of each other.* At this model scale and corpus size that can be a
  true result. Before concluding logits don't matter, check the one condition where they
  reliably do: restrict eval to the longest quartile of completions.

**The SME question for the verdict:** your next project has a 70B teacher behind an API that
returns text only. Which row of this table is your plan, what will it cost, and what evidence
from *this* run says the quality is acceptable?

## Exercises

1. **Sampled vs greedy SeqKD.** Regenerate at T=1.0 and retrain. Kim & Rush used the mode;
   modern practice often samples. Which wins here, on which metric, and does the entropy gap
   against cached-logit close?
2. **The `seq_kd=True` cross-check.** Run TRL's GKD trainer with `seq_kd=True, lmbda=1.0` on
   the same prompts and compare to your hand-rolled arm. They should land close; where they
   differ, read the trainer source to find the decision you made differently.
3. **Rationale distillation.** Prepend teacher-generated step-by-step rationales to the
   completions (generate them with a "think step by step" prompt) and SFT. Does the student
   improve on held-out prompts *without* rationales at inference?
4. **Matched-compute referee.** Recompute Part C's table at matched *total* compute (teacher
   generation + student training FLOPs) instead of matched student steps. Does the ranking
   change? This is the comparison the SeqKD papers rarely show.